In [15]:
!pip install sqlalchemy pymysql openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# DB mới
username = "local"
password = "dung2006!"
host = "127.0.0.1"
port = "3306"
database = "midterm_proj"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

print("Connected successfully!")

Connected successfully!


In [17]:
query = """
SELECT 
    f.firm_id,
    f.ticker,
    ff.fiscal_year AS year,
    ff.current_liabilities,
    ff.total_assets,
    ff.total_liabilities,
    ff.total_equity,
    ff.net_sales,
    ff.net_income,
    ff.net_operating_income,

    fm.shares_outstanding,
    fm.share_price,
    fm.market_value_equity,
    fm.eps_basic,

    fc.net_cfo,
    fc.net_cfi,
    fc.capex,

    fo.state_own,
    fo.foreign_own,

    fi.product_innovation,
    fi.process_innovation,

    meta.employees_count,
    meta.firm_age

FROM fact_financial_year ff
LEFT JOIN dim_firm f 
    ON ff.firm_id = f.firm_id

LEFT JOIN fact_market_year fm
    ON ff.firm_id = fm.firm_id
    AND ff.fiscal_year = fm.fiscal_year
    AND ff.snapshot_id = fm.snapshot_id

LEFT JOIN fact_cashflow_year fc
    ON ff.firm_id = fc.firm_id
    AND ff.fiscal_year = fc.fiscal_year
    AND ff.snapshot_id = fc.snapshot_id

LEFT JOIN fact_ownership_year fo
    ON ff.firm_id = fo.firm_id
    AND ff.fiscal_year = fo.fiscal_year
    AND ff.snapshot_id = fo.snapshot_id

LEFT JOIN fact_innovation_year fi
    ON ff.firm_id = fi.firm_id
    AND ff.fiscal_year = fi.fiscal_year
    AND ff.snapshot_id = fi.snapshot_id

LEFT JOIN fact_firm_year_meta meta
    ON ff.firm_id = meta.firm_id
    AND ff.fiscal_year = meta.fiscal_year
    AND ff.snapshot_id = meta.snapshot_id

WHERE ff.fiscal_year BETWEEN 2019 AND 2024
AND ff.snapshot_id = (
    SELECT MAX(snapshot_id)
    FROM fact_financial_year
)
"""

df = pd.read_sql(query, engine)

print("Total rows:", len(df))
df.head()


Total rows: 100


,firm_id,ticker,year,current_liabilities,total_assets,total_liabilities,total_equity,net_sales,net_income,net_operating_income,...,eps_basic,net_cfo,net_cfi,capex,state_own,foreign_own,product_innovation,process_innovation,employees_count,firm_age
0,1,AAM,2020,1.550367e+10,2.108193e+11,1.671828e+10,1.941010e+11,1.226512e+11,-1.196735e+10,-3.709323e+09,...,-1145.0,-2.431145e+10,3.434959e+10,1.370951e+09,0.0,0.0117,1,1,329.0,18.0
1,1,AAM,2021,5.544333e+09,2.010875e+11,6.761018e+09,1.943265e+11,1.341103e+11,2.255100e+08,-4.321664e+09,...,22.0,4.531249e+10,-2.287699e+09,7.460727e+08,0.0,0.0160,0,1,159.0,19.0
2,1,AAM,2022,7.937232e+09,2.185795e+11,9.043272e+09,2.095363e+11,2.120078e+11,1.689976e+10,1.199933e+10,...,1455.0,-3.970805e+09,-4.805325e+10,3.217644e+09,0.0,0.0116,0,1,165.0,20.0
3,1,AAM,2023,9.148047e+09,2.128573e+11,1.022154e+10,2.026358e+11,1.392592e+11,7.031509e+08,-9.834918e+08,...,40.0,-2.423085e+10,4.400175e+10,2.380000e+08,0.0,0.0103,0,1,176.0,21.0
4,1,AAM,2024,4.931214e+09,2.023532e+11,6.013144e+09,1.963401e+11,1.534833e+11,-6.295684e+09,-1.312532e+10,...,-602.0,3.404372e+10,5.443430e+09,1.050000e+08,0.0,0.0103,0,1,176.0,22.0


In [18]:
print("Columns:")
print(df.columns)

print("\nNumber of firms:", df['firm_id'].nunique())
print("Years:", sorted(df['year'].unique()))

Columns:
Index(['firm_id', 'ticker', 'year', 'current_liabilities', 'total_assets',
       'total_liabilities', 'total_equity', 'net_sales', 'net_income',
       'net_operating_income', 'shares_outstanding', 'share_price',
       'market_value_equity', 'eps_basic', 'net_cfo', 'net_cfi', 'capex',
       'state_own', 'foreign_own', 'product_innovation', 'process_innovation',
       'employees_count', 'firm_age'],
      dtype='object')

Number of firms: 20
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [19]:
import pandas as pd
import numpy as np

# Chuẩn hóa tên cột và sắp xếp đúng grain firm-year
df.columns = df.columns.str.lower()

if 'fiscal_year' in df.columns and 'year' not in df.columns:
    df['year'] = df['fiscal_year']
if 'net_cfi' in df.columns and 'cashflow_investing' not in df.columns:
    df['cashflow_investing'] = df['net_cfi']
if 'shares_outstanding' in df.columns and 'total_share_outstanding' not in df.columns:
    df['total_share_outstanding'] = df['shares_outstanding']
if {'product_innovation', 'process_innovation'}.issubset(df.columns) and 'innovation_dummy' not in df.columns:
    df['innovation_dummy'] = (
        (df['product_innovation'].fillna(0) == 1)
        | (df['process_innovation'].fillna(0) == 1)
    ).astype(int)

sort_keys = [c for c in ['firm_id', 'year'] if c in df.columns]
if sort_keys:
    df = df.sort_values(sort_keys).reset_index(drop=True)

# =============================
# HELPER FUNCTIONS
# =============================

def safe_ratio(num, den):
    num = pd.to_numeric(num, errors='coerce')
    den = pd.to_numeric(den, errors='coerce')
    return pd.Series(
        np.where(den.notna() & (den != 0), num / den, np.nan),
        index=num.index
    )


def safe_pct_diff(actual, expected, base=None):
    actual = pd.to_numeric(actual, errors='coerce')
    expected = pd.to_numeric(expected, errors='coerce')
    if base is None:
        base = expected.abs()
    else:
        base = pd.to_numeric(base, errors='coerce').abs()

    return pd.Series(
        np.where(base.notna() & (base != 0), (actual - expected).abs() / base, np.nan),
        index=actual.index
    )


qc = {}
rule_meta = {}


def add_rule(name, condition, severity, category, description):
    if not isinstance(condition, pd.Series):
        condition = pd.Series(condition, index=df.index)
    qc[name] = condition.fillna(False).astype(bool)
    rule_meta[name] = {
        'severity': severity,
        'category': category,
        'description': description
    }


# =============================
# CALCULATED VARIABLES
# =============================
if {'total_liabilities', 'total_assets'}.issubset(df.columns):
    df['calc_debt_ratio'] = safe_ratio(df['total_liabilities'], df['total_assets'])

if {'net_income', 'total_assets'}.issubset(df.columns):
    df['calc_roa'] = safe_ratio(df['net_income'], df['total_assets'])

if {'net_income', 'net_sales'}.issubset(df.columns):
    df['calc_net_margin'] = safe_ratio(df['net_income'], df['net_sales'])

if {'shares_outstanding', 'share_price'}.issubset(df.columns):
    df['calc_market_cap'] = df['shares_outstanding'] * df['share_price']

# =============================
# KEY / STRUCTURE CHECKS
# =============================
if 'firm_id' in df.columns:
    add_rule('missing_firm_id', df['firm_id'].isna(), 'critical', 'key', 'Thiếu firm_id')
if 'year' in df.columns:
    add_rule('missing_year', df['year'].isna(), 'critical', 'key', 'Thiếu year')
if 'ticker' in df.columns:
    add_rule('missing_ticker', df['ticker'].isna(), 'business', 'key', 'Thiếu ticker')
if {'firm_id', 'year'}.issubset(df.columns):
    add_rule(
        'duplicate_firm_year',
        df.duplicated(subset=['firm_id', 'year'], keep=False),
        'critical',
        'structure',
        'Trùng grain firm_id-year'
    )

# =============================
# COMPLETENESS CHECKS
# =============================
core_cols = [c for c in ['net_sales', 'total_assets', 'total_equity'] if c in df.columns]
if core_cols:
    add_rule(
        'missing_core',
        df[core_cols].isna().any(axis=1),
        'critical',
        'completeness',
        'Thiếu ít nhất một trường core trong bộ cột chính'
    )

for col in ['net_sales', 'total_assets', 'net_income']:
    if col in df.columns:
        add_rule(
            f'missing_{col}',
            df[col].isna(),
            'business',
            'completeness',
            f'Thiếu dữ liệu ở cột {col}'
        )

# =============================
# RANGE / SANITY CHECKS
# =============================
if 'net_sales' in df.columns:
    add_rule('net_sales_negative', df['net_sales'] < 0, 'business', 'range', 'Net sales âm')
if 'total_assets' in df.columns:
    add_rule('assets_negative', df['total_assets'] < 0, 'critical', 'range', 'Total assets âm')
if 'total_liabilities' in df.columns:
    add_rule('total_liabilities_negative', df['total_liabilities'] < 0, 'critical', 'range', 'Total liabilities âm')
if 'current_liabilities' in df.columns:
    add_rule('current_liabilities_negative', df['current_liabilities'] < 0, 'business', 'range', 'Current liabilities âm')
if {'total_equity', 'total_assets'}.issubset(df.columns):
    add_rule(
        'equity_too_negative',
        df['total_equity'] < -df['total_assets'],
        'critical',
        'range',
        'Total equity âm quá mức so với total assets'
    )
if 'shares_outstanding' in df.columns:
    add_rule(
        'shares_outstanding_invalid',
        df['shares_outstanding'].notna() & (df['shares_outstanding'] <= 0),
        'business',
        'range',
        'Shares outstanding <= 0'
    )
if 'capex' in df.columns:
    add_rule(
        'capex_negative',
        df['capex'].notna() & (df['capex'] < 0),
        'warning',
        'range',
        'Capex âm - cần kiểm tra lại định nghĩa dấu của dữ liệu'
    )

# Ownership checks
if 'state_own' in df.columns:
    add_rule(
        'state_own_out_of_range',
        df['state_own'].notna() & ((df['state_own'] < 0) | (df['state_own'] > 1)),
        'business',
        'ownership',
        'State ownership ngoài khoảng [0,1]'
    )
if 'foreign_own' in df.columns:
    add_rule(
        'foreign_own_out_of_range',
        df['foreign_own'].notna() & ((df['foreign_own'] < 0) | (df['foreign_own'] > 1)),
        'business',
        'ownership',
        'Foreign ownership ngoài khoảng [0,1]'
    )
if {'state_own', 'foreign_own'}.issubset(df.columns):
    add_rule(
        'ownership_sum_gt_1',
        df['state_own'].notna() & df['foreign_own'].notna() & ((df['state_own'] + df['foreign_own']) > 1),
        'warning',
        'ownership',
        'Tổng state_own + foreign_own vượt 100%'
    )

# =============================
# FINANCIAL LOGIC CHECKS
# =============================
if {'total_assets', 'total_liabilities', 'total_equity'}.issubset(df.columns):
    bs_gap_pct = safe_pct_diff(
        df['total_assets'],
        df['total_liabilities'] + df['total_equity'],
        base=df['total_assets']
    )
    add_rule(
        'balance_sheet_error_3pct',
        bs_gap_pct > 0.03,
        'critical',
        'accounting',
        'Assets khác Liabilities + Equity quá 3%'
    )
    add_rule(
        'balance_sheet_gap_over_1pct',
        bs_gap_pct > 0.01,
        'warning',
        'accounting',
        'Assets khác Liabilities + Equity quá 1%'
    )

if {'market_value_equity', 'calc_market_cap'}.issubset(df.columns):
    market_cap_diff = safe_pct_diff(
        df['market_value_equity'],
        df['calc_market_cap'],
        base=df['market_value_equity']
    )
    add_rule(
        'market_cap_error',
        market_cap_diff > 0.05,
        'business',
        'market',
        'Market value equity khác shares_outstanding * share_price quá 5%'
    )

if {'net_sales', 'net_income'}.issubset(df.columns):
    add_rule(
        'income_without_net_sales',
        df['net_sales'].fillna(0).eq(0) & df['net_income'].fillna(0).ne(0),
        'business',
        'logic',
        'Net income khác 0 trong khi net sales = 0'
    )

if {'innovation_dummy', 'rd_expense'}.issubset(df.columns):
    add_rule(
        'innovation_without_rd',
        (df['innovation_dummy'] == 1) & (df['rd_expense'].isna() | (df['rd_expense'] <= 0)),
        'warning',
        'logic',
        'Có innovation nhưng rd_expense thiếu hoặc <= 0'
    )

if {'capex', 'cashflow_investing'}.issubset(df.columns):
    add_rule(
        'capex_inconsistency',
        df['capex'].notna() & (df['capex'] > 0) & df['cashflow_investing'].notna() & (df['cashflow_investing'] >= 0),
        'warning',
        'cashflow',
        'Capex dương nhưng cashflow_investing không âm - cần kiểm tra định nghĩa dấu hoặc dữ liệu'
    )

# =============================
# RATIO / OUTLIER CHECKS
# =============================
if 'calc_roa' in df.columns:
    add_rule(
        'extreme_roa',
        df['calc_roa'].abs() > 2,
        'warning',
        'ratio',
        'ROA tuyệt đối vượt 2 lần'
    )

if 'calc_debt_ratio' in df.columns:
    add_rule(
        'debt_ratio_above_1_5',
        df['calc_debt_ratio'] > 1.5,
        'warning',
        'ratio',
        'Debt ratio vượt 1.5'
    )

# =============================
# TIME SERIES CHECKS
# =============================
if {'firm_id', 'year', 'net_sales'}.issubset(df.columns):
    df = df.sort_values(['firm_id', 'year']).reset_index(drop=True)
    net_sales_lag = df.groupby('firm_id')['net_sales'].shift(1)
    df['net_sales_growth'] = safe_ratio(df['net_sales'] - net_sales_lag, net_sales_lag)

    add_rule(
        'net_sales_growth_abnormal',
        net_sales_lag.abs().gt(1) & ((df['net_sales_growth'] > 5) | (df['net_sales_growth'] < -0.95)),
        'warning',
        'timeseries',
        'Net sales growth bất thường so với năm trước'
    )
    add_rule(
        'extreme_growth',
        net_sales_lag.abs().gt(1) & ((df['net_sales_growth'] > 10) | (df['net_sales_growth'] < -0.99)),
        'warning',
        'timeseries',
        'Net sales growth cực đoan'
    )
    add_rule(
        'net_sales_jump',
        net_sales_lag.abs().gt(1) & (df['net_sales_growth'].abs() > 5),
        'warning',
        'timeseries',
        'Net sales nhảy quá mạnh so với năm trước'
    )

if {'firm_id', 'year', 'total_assets'}.issubset(df.columns):
    df = df.sort_values(['firm_id', 'year']).reset_index(drop=True)
    assets_lag = df.groupby('firm_id')['total_assets'].shift(1)
    df['assets_growth'] = safe_ratio(df['total_assets'] - assets_lag, assets_lag)

    add_rule(
        'assets_growth_abnormal',
        assets_lag.abs().gt(1) & (df['assets_growth'].abs() > 3),
        'warning',
        'timeseries',
        'Total assets tăng/giảm bất thường so với năm trước'
    )

# =============================
# CREATE QC TABLES
# =============================
qc_df = pd.DataFrame(qc, index=df.index)
rule_meta_df = pd.DataFrame.from_dict(rule_meta, orient='index').reset_index().rename(columns={'index': 'rule'})

total_obs = len(df)
results = []

for rule in qc_df.columns:
    flagged = int(qc_df[rule].sum())
    accuracy = ((total_obs - flagged) / total_obs * 100) if total_obs else np.nan
    results.append({
        'rule': rule,
        'flagged_rows': flagged,
        'flag_rate_%': round(flagged / total_obs * 100, 2) if total_obs else np.nan,
        'accuracy_%': round(accuracy, 2) if total_obs else np.nan
    })

summary = (
    pd.DataFrame(results)
    .merge(rule_meta_df, on='rule', how='left')
    .sort_values(['severity', 'flagged_rows', 'rule'], ascending=[True, False, True])
    .reset_index(drop=True)
)

summary


,rule,flagged_rows,flag_rate_%,accuracy_%,severity,category,description
0,current_liabilities_negative,0,0.0,100.0,business,range,Current liabilities âm
1,foreign_own_out_of_range,0,0.0,100.0,business,ownership,"Foreign ownership ngoài khoảng [0,1]"
2,income_without_net_sales,0,0.0,100.0,business,logic,Net income khác 0 trong khi net sales = 0
3,market_cap_error,0,0.0,100.0,business,market,Market value equity khác shares_outstanding * ...
4,missing_net_income,0,0.0,100.0,business,completeness,Thiếu dữ liệu ở cột net_income
5,missing_net_sales,0,0.0,100.0,business,completeness,Thiếu dữ liệu ở cột net_sales
6,missing_ticker,0,0.0,100.0,business,key,Thiếu ticker
7,missing_total_assets,0,0.0,100.0,business,completeness,Thiếu dữ liệu ở cột total_assets
8,net_sales_negative,0,0.0,100.0,business,range,Net sales âm
9,shares_outstanding_invalid,0,0.0,100.0,business,range,Shares outstanding <= 0


In [20]:
# Tổng quan toàn bộ bộ QC
metrics = []
for severity in ['critical', 'business', 'warning']:
    rules_in_scope = rule_meta_df.loc[rule_meta_df['severity'] == severity, 'rule'].tolist()
    if not rules_in_scope:
        continue

    total_checks = total_obs * len(rules_in_scope)
    total_flags = int(qc_df[rules_in_scope].sum().sum())
    quality_score = ((total_checks - total_flags) / total_checks * 100) if total_checks else np.nan

    metrics.append({
        'severity': severity,
        'rules': len(rules_in_scope),
        'total_flags': total_flags,
        'quality_score_%': round(quality_score, 2) if total_checks else np.nan
    })

overall_total_checks = total_obs * len(qc_df.columns)
overall_total_flags = int(qc_df.sum().sum())
overall_quality_score = ((overall_total_checks - overall_total_flags) / overall_total_checks * 100) if overall_total_checks else np.nan

print('Total observations:', total_obs)
print('Total rules:', len(qc_df.columns))
print('Overall Quality Score:', round(overall_quality_score, 2), '%')

pd.DataFrame(metrics)


Total observations: 100
Total rules: 29
Overall Quality Score: 99.21 %


,severity,rules,total_flags,quality_score_%
0,critical,8,0,100.0
1,business,11,0,100.0
2,warning,10,23,97.7


In [21]:
firm_summary = []

critical_rules = rule_meta_df.loc[rule_meta_df['severity'] == 'critical', 'rule'].tolist()
warning_rules = rule_meta_df.loc[rule_meta_df['severity'] == 'warning', 'rule'].tolist()

for firm, group in df.groupby('firm_id', dropna=False):
    sub_qc = qc_df.loc[group.index]
    total_checks = len(group) * len(qc_df.columns)
    total_flags = int(sub_qc.sum().sum())
    overall_quality_score = ((total_checks - total_flags) / total_checks * 100) if total_checks else np.nan

    critical_flag_rate = np.nan
    if critical_rules:
        critical_total = len(group) * len(critical_rules)
        critical_flag_rate = (sub_qc[critical_rules].sum().sum() / critical_total * 100) if critical_total else np.nan

    warning_flag_rate = np.nan
    if warning_rules:
        warning_total = len(group) * len(warning_rules)
        warning_flag_rate = (sub_qc[warning_rules].sum().sum() / warning_total * 100) if warning_total else np.nan

    firm_summary.append({
        'firm_id': firm,
        'rows': len(group),
        'overall_quality_score_%': round(overall_quality_score, 2) if total_checks else np.nan,
        'critical_flag_rate_%': round(critical_flag_rate, 2) if pd.notna(critical_flag_rate) else np.nan,
        'warning_flag_rate_%': round(warning_flag_rate, 2) if pd.notna(warning_flag_rate) else np.nan
    })

pd.DataFrame(firm_summary).sort_values('overall_quality_score_%')


,firm_id,rows,overall_quality_score_%,critical_flag_rate_%,warning_flag_rate_%
0,1,5,97.93,0.0,6.0
4,5,5,97.93,0.0,6.0
15,16,5,97.93,0.0,6.0
9,10,5,97.93,0.0,6.0
6,7,5,98.62,0.0,4.0
14,15,5,98.62,0.0,4.0
19,20,5,98.62,0.0,4.0
7,8,5,99.31,0.0,2.0
12,13,5,99.31,0.0,2.0
10,11,5,99.31,0.0,2.0


In [22]:
qc_records = []

for rule in qc_df.columns:
    mask = qc_df[rule]

    if mask.sum() == 0:
        continue

    meta = rule_meta.get(rule, {})
    error_type = meta.get('severity', 'unknown')
    message = meta.get('description', '')

    error_rows = df.loc[mask]

    for _, row in error_rows.iterrows():
        qc_records.append({
            'ticker': row.get('ticker'),
            'fiscal_year': row.get('year'),
            'field_name': rule,
            'error_type': error_type,
            'message': message
        })

qc_report = pd.DataFrame(qc_records)

# save file
import os
os.makedirs("outputs", exist_ok=True)

qc_report.to_csv("outputs/qc_report.csv", index=False)

qc_report.head()

,ticker,fiscal_year,field_name,error_type,message
0,AAM,2020,capex_inconsistency,warning,Capex dương nhưng cashflow_investing không âm ...
1,AAM,2023,capex_inconsistency,warning,Capex dương nhưng cashflow_investing không âm ...
2,AAM,2024,capex_inconsistency,warning,Capex dương nhưng cashflow_investing không âm ...
3,ACG,2022,capex_inconsistency,warning,Capex dương nhưng cashflow_investing không âm ...
4,DCL,2021,capex_inconsistency,warning,Capex dương nhưng cashflow_investing không âm ...
